# PyMuPDF (detect & extract tables, images, text)

- regular table (lines around) works well, not for complicated format like several lines in a block
- extract the image out, could used for further analysis with page number indexed

## extract table from pdf

In [1]:
#%pip install pymupdf
#%pip install pandas

In [16]:
import pymupdf
#47s for find tables for 266 pages' document
#better use RAG to load and fine desired table at first
doc = pymupdf.open("example.pdf")
page = doc[4]  # this is the first page
tabs = page.find_tables()
print(f"{len(tabs.tables)} table(s) on {page}")

1 table(s) on page 4 of example.pdf


In [17]:
tab = tabs[0]
type(tab)
#tab.bbox  # bounding box of the full table
#tab.cells[0]  # top-left cell
#tab.cells[-1]  # bottom-right cell
#tab.row_count, tab.col_count  # row and column counts
#tab.page  # backreference to the page
#for line in tab.extract():  # print cell text for each row
    #print(line)

pymupdf.table.Table

In [18]:
#draw the bbox
r = pymupdf.Rect(tab.bbox)
shape=page.new_shape()
shape.draw_line(r.tl, r.tr)
shape.draw_line(r.tl, r.bl)
shape.draw_line(r.br, r.tr)
shape.draw_line(r.br, r.bl)
shape.finish(color=(1,1,0), width=1) #aliceblue
shape.commit()
doc.save("x.pdf")

In [4]:
header = tab.header  # table header object
#header.bbox  # header bbox
#header.cells[0]  # leftmost header cell
#header.cells[-1]  # rightmost header cell
#header.external  # header is part of the table itself!
header.names  # these are the column names

['Magnitude', 'Description']

In [5]:
df = tab.to_pandas()  # convert to pandas DataFrame
#df.to_excel(f"{doc.name}-{page.number}.xlsx")
df

,Magnitude,Description
0,Major positive benefit\nor change,Refers to significant improvements in baseline...
1,Moderate positive\nbenefit or change,Refers to significant improvements in local ba...
2,Minor positive benefit\nor change,Refers to minor positive benefit or change imp...
3,Slight positive benefit\nor change,Refers to slight positive benefit or change im...
4,No change/status\nquo,Refers to no change/status quo implies that ch...
5,Slight negative\ndisadvantage or\nchange,Refers to changes in baseline conditions that ...
6,Minor negative\ndisadvantage or\nchange,Refers to negative changes to baseline conditi...
7,Moderate negative\ndisadvantage or\nchange,Refers to significant adverse changes in local...
8,Major negative\ndisadvantage or\nchange,Refers to significant adverse changes in basel...


## to - markdown (with image/chunks/RAG documents)

In [6]:
#%pip install pymupdf4llm

- Support for multi-column pages
- Support for image and vector graphics extraction (and inclusion of references in the MD text)
- Support for page chunking output
- Direct support for output as LlamaIndex Documents

In [7]:
import pymupdf4llm
output = pymupdf4llm.to_markdown("example.pdf", write_images=True)
# write markdown string to some file
md = open('test.md', 'w')
md.write(output)
md.close()

Processing example.pdf...
[                                        ] (0/6=====[======                                  ] (1/6)

======[=============                           ] (2/======[====================                    ] (3/6=====[==========================              ] (4/6======[=================================       ] (5/======[========================================] (6/6]


The images will be saved to the local folder. The markdown is directly referenced to the image.

In [8]:
import pymupdf4llm
output = pymupdf4llm.to_markdown("example.pdf", page_chunks=True)

Processing example.pdf...
[                                        ] (0/6=====[======                                  ] (1/6)

======[=============                           ] (2/======[====================                    ] (3/6=====[==========================              ] (4/6======[=================================       ] (5/======[========================================] (6/6]


chunk function gives list of documents like RAG but not that accuracy.

In [9]:
#%pip install llama_index

In [10]:
import pymupdf4llm
llama_reader = pymupdf4llm.LlamaMarkdownReader()
llama_docs = llama_reader.load_data("example.pdf")

Successfully imported LlamaIndex
Processing example.pdf...
[                                        ] (0/1=======================================[========================================] (1/1]
Processing example.pdf...
[                                        ] (0/1=======================================[========================================] (1/1]
Processing example.pdf...
[                                        ] (0/1=======================================[========================================] (1/1]
Processing example.pdf...
[                                        ] (0/1=======================================[========================================] (1/1]
Processing example.pdf...
[                                        ] (0/1=======================================[========================================] (1/1]
Processing example.pdf...
[                                        ] (0/1=======================================[========================================] (1/1]


In [11]:
llama_docs

[Document(id_='fb76b39d-6f7e-428b-8755-73abceb484ed', embedding=None, metadata={'format': 'PDF 1.3', 'title': '', 'author': 'Holly Siow', 'subject': '', 'keywords': '', 'creator': 'Microsoft® Word for Microsoft 365', 'producer': 'macOS 版本15.1.1（版号24B91） Quartz PDFContext, AppendMode 1.1', 'creationDate': "D:20241204030430Z00'00'", 'modDate': "D:20241204033554Z00'00'", 'trapped': '', 'encryption': None, 'page': 1, 'total_pages': 6, 'file_path': 'example.pdf'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Given the ambiguity in the nature of assessing the “magnitude” component, we use the following\ncriteria, tabulated in Table 3.4, to aid the assessment.\n\n**Table 3.4. Description of the value of magnitudes in RIAM method**\n\n**Magnitude** **Description**\n\nRefers to significant improvements in baseline conditions and a significant\n\nMajor positive benefit\n\nreduction of stress o

## examples - find species name

- load pdf via pypdf in langchain (20s)
- create database
- find desired tables' page number
- extract the table by pymupdf (2.5s)
- summarize the name

#### format required:

- name in table, column names 'species name' (lower- or upper- case)

In [13]:
#%pip install langchain-community

In [ ]:
import re
from langchain_community.document_loaders import PyPDFLoader
import pymupdf

file_path = ('data.pdf')
loader = PyPDFLoader(file_path)
pages = loader.load_and_split() #text_splitter=text_splitter)

## species name
species_name_pattern = re.compile(r'species name', re.IGNORECASE)
species_list = []
for i in pages:
    if re.search(species_name_pattern, i.page_content):
        species_list.append(i.metadata['page'])

species_list = sorted(list(set(species_list)))

In [28]:
species_list

[14, 58, 59, 60, 66, 67, 71, 72, 74, 75, 77, 86, 87]

In [95]:
## species name
species_name_pattern = re.compile(r'species name', re.IGNORECASE)
name_output_list = []

for i in species_list:
    page = doc[i]
    tabs = page.find_tables()
    num = len(tabs.tables)
    print(f"{num} table(s) on {page}")
    
    for j in range(0, num):
        tab = tabs[j]
        header = tab.header #list
        header_list = [str(item) for item in header.names]
        header_string = ' '.join(header_list)
        
        if re.search(species_name_pattern, header_string):
            df = tab.to_pandas()
            # Normalize the column names to lowercase
            df.columns = df.columns.str.lower()
            names = df['species name'].values.tolist()
            tmp = [i for i in names if i is not None]
            tmp1 = [string.replace('\n', ' ') for string in tmp]
            name_output_list.extend(tmp1)

1 table(s) on page 14 of data.pdf
1 table(s) on page 58 of data.pdf
1 table(s) on page 59 of data.pdf
2 table(s) on page 60 of data.pdf
1 table(s) on page 66 of data.pdf
1 table(s) on page 67 of data.pdf
1 table(s) on page 71 of data.pdf
1 table(s) on page 72 of data.pdf
1 table(s) on page 74 of data.pdf
1 table(s) on page 75 of data.pdf
1 table(s) on page 77 of data.pdf
2 table(s) on page 86 of data.pdf
1 table(s) on page 87 of data.pdf


In [96]:
name_output_list

['Ampelocissus elegans',
 'Ampelocissus gracilis',
 'Aphanamixis polystachya',
 'Archidendron contortum',
 'Archidendron jiringa',
 'Ardisia elliptica',
 'Artabotrys maingayi',
 'Artabotrys suaveolens',
 'Artocarpus lacucha',
 'Baccaurea motleyana',
 'Bridelia stipularis',
 'Callicarpa longifolia',
 'Calophyllum inophyllum',
 'Causonis trifolia',
 'Chassalia curviflora',
 'Cissus repens',
 'Clerodendrum villosum',
 'Cratoxylum maingayi',
 'Cyathea latebrosa',
 'Cyclosorus opulentus',
 'Cyclosorus polycarpus',
 'Cyrtococcum accrescens',
 'Cyrtococcum patens',
 'Dalbergia junghuhnii',
 'Dendrotrophe varians',
 'Dioscorea cf polyclados',
 'Dissochaeta sp.',
 'Dracaena cantleyi',
 'Embelia canescens',
 'Enkleia malaccensis',
 'Eurycoma longifolia',
 'Ficus apiocarpa',
 'Ficus aurata',
 'Ficus sagittata',
 'Ficus superba',
 'Ficus vasculosa',
 'Glochidion zeylanicum var. zeylanicum',
 'Gnetum latifolium',
 'Goniophlebium percussum',
 'Grenacheria amentacea',
 'Guioa pubescens',
 'Gynochthod

In [ ]:
1 - 4/len(name_output_list) #only 4 false values are included, all true include
# precison tp/tp+fp
# 3 due to format
# 1 due to pdf read-in error

0.972027972027972

One bug still:
    for page 78 (index 77), to exclude species group name
    set a filter technique:
        the name is not None
        for that row, the rest of columns' value are not all None

while not worth it, cannot make sure the right species name fit the format (non all None in the row)

## example - status

#### format required:

- status in table, column names 'Conservation Status', first column in table

In [100]:
import re
from langchain_community.document_loaders import PyPDFLoader
import pymupdf

file_path = ('data.pdf')
loader = PyPDFLoader(file_path)
pages = loader.load_and_split() #text_splitter=text_splitter)
status_pattern = r'Conservation Status'
status_list = []
for i in pages:
    if re.search(status_pattern, i.page_content):
        status_list.append(i.metadata['page'])

status_list = sorted(list(set(status_list)))

In [101]:
status_list

[13, 46, 58, 59, 60, 86]

In [107]:
## species name
status_pattern = r'Conservation Status'
status_output_list = []

for i in status_list:
    page = doc[i]
    tabs = page.find_tables()
    num = len(tabs.tables)
    print(f"{num} table(s) on {page}")
    
    for j in range(0, num):
        tab = tabs[j]
        header = tab.header #list
        
        if 'Conservation Status' == header.names[0]:
            df = tab.to_pandas()
            # Normalize the column names to lowercase
            df.columns = df.columns.str.lower()
            names = df['conservation status'].values.tolist()
            tmp = [i for i in names if i is not None]
            tmp1 = [string.replace('\n', ' ') for string in tmp]
            status_output_list.extend(tmp1)

0 table(s) on page 13 of data.pdf
1 table(s) on page 46 of data.pdf
1 table(s) on page 58 of data.pdf
1 table(s) on page 59 of data.pdf
2 table(s) on page 60 of data.pdf
2 table(s) on page 86 of data.pdf


In [108]:
status_output_list

['Global',
 'Extinct (EX)',
 'Extinct in the Wild (EW)',
 'Critically Endangered (CR)',
 'Endangered (EN)',
 'Vulnerable (VU)',
 'Near Threatened (NT)',
 'Least Concern (LC)',
 'Data Deficient (DD)',
 'Not Evaluated (NE)',
 'Local',
 'Presumed Nationally Extinct (NE)',
 'Critically Endangered (CR)',
 'Endangered (EN)',
 'Vulnerable (VU)']

# Others

#### detection model + OCR extraction

- detection model:

  - yolo8:
    - https://iamrajatroy.medium.com/document-intelligence-series-part-1-table-detection-with-yolo-1fa0a198fd7
    - https://huggingface.co/microsoft/table-transformer-structure-recognition

  - DETR (detection transformer from microsoft):
    - https://huggingface.co/microsoft/table-transformer-detection
    - https://huggingface.co/docs/transformers/main/en/model_doc/table-transformer
    - https://iamrajatroy.medium.com/document-intelligence-series-part-2-transformer-for-table-detection-extraction-80a52486fa3

- OCR:
  - pytesseract (Google's Tesseract-OCR) -- could handle some table without border like 3.png, but also fail to extract some regular table


In [33]:
#pytesseract
#%pip install pytesseract
from PIL import Image
import pytesseract
from pytesseract import Output
img = Image.open('3.png')
#ext_df = pytesseract.image_to_data(img, output_type=Output.DATAFRAME, config="--psm 6 --oem 3")
print(pytesseract.image_to_string(Image.open('3.png')))

 

Range Impact

116 to 180 Major positive change/impact

 

 

 

81 to 115 Moderate positive change/impact
37 to 80 Minor positive change/impact
7 to 36 Slight positive impact

 

